In [ ]:
import numpy as np
from regpy.solvers import Solver

## The Jacobi iteration implemented as regpy.solver.Solver

Recall that the Jacobi method for solving a quadratic operator equation $Ax=b$ consists in splitting the matrix $A= D+R$ into a diagonal matrix $D$ and a remainder matrix $R$ and performing the fixed point iteration

$$
x_{n+1} = D^{-1} (b - Rx_n).
$$

In [ ]:
class Jacobi(Solver):
    def __init__(self, A: np.ndarray, b: np.ndarray, x0: np.ndarray | None = None):
        super().__init__(x=x0 if x0 is not None else np.zeros_like(b), y=None)
        # As the operator values y_n= Ax_n are not needed for the Jacobi method, 
        # we do not compute them here and set self.y to None.
        self.b = b
        self.diagA = np.diag(A)
        self.R = A - np.diag(self.diagA)

    def _next(self):
        self.x = (self.b - self.R @ self.x) / self.diagA

Create a Jacobi solver for some problem instance.

In [ ]:
A = np.array([[4, -1, 0, 0],
              [-1, 4, -1, 0],   
              [0, -1, 4, -1],
              [0, 0, -1, 3]], dtype=float)
b = np.array([15, 10, 10, 10], dtype=float)
xref = np.linalg.solve(A, b)
Jacobi_solver = Jacobi(A, b)


To produce the iterates of the Jacobi method, we create a generator using `iter` and repeatedly calling next on this generator. 

In [ ]:
it = iter(Jacobi_solver)
for i in range(10):
    x_n, _ = next(it)
    print(f"Iteration {i+1}: x_n = {x_n}, error = {np.linalg.norm(x_n - xref)}")

Of course, solvers can also be run and inspected manually, e.g.,  by repeatedly clicking a cell. This is often useful during development stage. 

In [ ]:
next(it)

Eventually, iterations should be stopped in a fully data driven manner by a `StopRule`. 

For the Jacobi iteration, one often uses the error bound
$$
\|x_{n}-x\|_{\infty} \leq \frac{q}{1-q}\|x_{n}-x_{n-1}\|_{\infty}, 
\qquad q:=\max_{j}\sum_{l\neq j} \left|\frac{a_{jl}}{a_{jj}}\right|,
$$
derived from the a-posteriori estimate in Banach's fixed point theorem 
provided $q<1$ and $n\geq 1$.

We implement a `StopRule` guaranteeing an absolute accuracy smaller than a given tolerance in the supremum norm. Note that this is only one of many useful stopping rules. Specifically, one can use different vector norms (e.g., if $q\geq 1$) or relative tolerances. `StopRule`s can also be used for monitoring. Their implementation may be more involved than that of the solver.
 
For these reasons, it is advantageous to keep the stop rules separate from the solver itself even if they are solver-specific. 

In [ ]:
from regpy.stoprules import StopRule


class JacobiStopping(StopRule):
    """Stop when the a-posteriori bound for the current iterate is below tol.

    Parameters
    ----------
    tol: float [default: 1e-6]
        Positive absolute tolerance for the error in the supremum norm.
    logging_level: int | str [default: "INFO"]
        Logging level passed to StopRule.

    The attached solver must be a Jacobi solver with contraction factor q < 1.
    The rule must be checked at every iteration. It needs two consecutive
    iterates, so it cannot certify the initial guess. 
    """

    def __init__(self, tol: float = 1e-6,
                 logging_level: int | str = "INFO") -> None:
        super().__init__(logging_level=logging_level)
        if not np.isfinite(tol) or tol <= 0:
            raise ValueError("tol must be finite and strictly positive.")
        self.tol = float(tol)
        self.q: float | None = None
        self.error_bound = np.inf
        self._previous_x: np.ndarray | None = None
        self._previous_step: int | None = None

    def _complete_init_with_solver(self, solver: Jacobi) -> None:
        """To avoid circular imports, this method is called by the solver 
        when the rule is attached. This allows the rule to access solver 
        attributes without importing the solver class here.
        Solver-dependent initializations such as the computation 
        of the contraction factor q is done here.
        """
        if not isinstance(solver, Jacobi):
            raise TypeError("JacobiStopping requires a Jacobi solver.")
        if np.any(solver.diagA == 0):
            raise ValueError("The Jacobi diagonal must be nonzero.")
        q = float(np.max(np.sum(
            np.abs(solver.R / solver.diagA[:, None]), axis=1
        )))
        if not np.isfinite(q) or q >= 1:
            raise ValueError(f"The error bound requires q < 1; got q={q}.")
        super()._complete_init_with_solver(solver)
        self.q = q
        self.error_bound = np.inf
        # A snapshot is essential: a solver may modify its iterate in place.
        self._previous_x = np.array(solver.x, copy=True)
        self._previous_step = solver.iteration_step_nr

    def _stop(self) -> bool:
        if self.solver is None or self._previous_x is None:
            raise RuntimeError("Attach the rule to a solver before checking it.")
        step = self.solver.iteration_step_nr
        if step == self._previous_step:
            # Includes the initial check; repeated checks do not create a
            # spurious zero difference. 
            return bool(self.error_bound < self.tol)
        if step != self._previous_step + 1:
            raise RuntimeError("JacobiStopping must be checked at every iteration.")

        change = np.linalg.norm(self.solver.x - self._previous_x, ord=np.inf)
        self.error_bound = float(self.q / (1 - self.q) * change)
        self._previous_x = np.array(self.solver.x, copy=True)
        self._previous_step = step
        return bool(self.error_bound < self.tol)

    def reset(self) -> None:
        super().reset()
        self.q = None
        self.error_bound = np.inf
        self._previous_x = None
        self._previous_step = None


Use a fresh solver to demonstrate automatic stopping. Calling `reset()` allows the rule to be reused.

Here we use `while_` to perform monitoring of the error. (This could also be delegated to the StopRule!). If only the final iterate is of interest, you can 
simply use `x, _ = solver.run(rule)`. 


In [ ]:
from matplotlib import pyplot as plt
solver = Jacobi(A, b)
rule = JacobiStopping(tol=1e-6)
errors = []
for x, _ in solver.while_(rule,max_it=1000):
    errors.append(np.linalg.norm(x - xref,ord=np.inf))

plt.semilogy(errors)
plt.xlabel("Iteration")
plt.ylabel("Error")
plt.title("Convergence of Jacobi Method")
plt.show()
